In [ ]:
%%capture
%pip install -U bitsandbytes
%pip install -U transformers
%pip install -U accelerate
%pip install -U peft
%pip install -U trl

In [ ]:
!pip install transformers accelerate huggingface-hub


In [ ]:
from google.colab import userdata

In [ ]:
from huggingface_hub import login
login(token=userdata.get('hugging_face_token'))

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from trl import setup_chat_format
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging,
                          LlamaForCausalLM)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)
from sklearn.preprocessing import MultiLabelBinarizer

## Load Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path= '/content/drive/MyDrive/Datasci_266_NLP/Final_Project/Baseline 20241107/'

In [ ]:
all_data = pd.read_csv(path+'all_data_all_lang.csv')
all_data.rename(columns={'Unnamed: 0':'all_indices'}, inplace=True)
all_data['fine-grained_roles_list']=all_data['fine-grained_roles'].apply(lambda x: [k.strip(" ").strip("'") for k in [x[1:-1].strip("'")][0].split(",")])
all_data['fine-grained_roles']=all_data['fine-grained_roles'].apply(lambda x: x[1:-1].replace("'",""))

In [ ]:
validation_data = pd.read_csv(path+'all_data_all_lang_validation.csv')
validation_data.rename(columns={'Unnamed: 0':'val_indices'}, inplace=True)

In [ ]:
train_indices=list(set(all_data['all_indices'].values) - set(validation_data['val_indices'].values))

val_indices=list(validation_data['val_indices'].values)

In [ ]:
#data splits
X_train = all_data[['entity_mention','article','main_role','fine-grained_roles']].iloc[train_indices]
X_val_= all_data[['entity_mention','article','main_role','fine-grained_roles','fine-grained_roles_list']].iloc[val_indices]


In [ ]:
X_train

,entity_mention,article,main_role,fine-grained_roles
0,Chinese,The World Needs Peacemaker Trump Again \n\n by...,Antagonist,Spy
1,China,The World Needs Peacemaker Trump Again \n\n by...,Antagonist,Instigator
2,Hamas,The World Needs Peacemaker Trump Again \n\n by...,Antagonist,Terrorist
3,Donald Trump,The World Needs Peacemaker Trump Again \n\n by...,Protagonist,"Peacemaker, Guardian"
4,Yermak,"Ukraine's Fate Will Be Decided In Coming Year,...",Antagonist,Incompetent
...,...,...,...,...
2528,Estados Unidos,A transição energética\n\nMultiplicam-se os fe...,Innocent,Victim
2529,Canadá,A transição energética\n\nMultiplicam-se os fe...,Innocent,Victim
2530,China,A transição energética\n\nMultiplicam-se os fe...,Innocent,Victim
2531,Pokrovsk,Rússia assume controlo de mais uma povoação no...,Innocent,Victim


In [ ]:
X_val_

,entity_mention,article,main_role,fine-grained_roles,fine-grained_roles_list
1414,रूस,प्रधानमंत्री नरेन्द्र मोदी और रूस के राष्ट्रपत...,Protagonist,Virtuous,[Virtuous]
1776,ओरबान,Viktor Orban in China: हंगरी के प्रधानमंत्री व...,Antagonist,Incompetent,[Incompetent]
2323,Comissão Europeia,UE transfere mais de 9 mil milhões de dólares ...,Protagonist,Virtuous,[Virtuous]
1389,रुबिज़ने,यूक्रेन ने निंदा की कि रूसी सेना ने रुबिज़ने औ...,Innocent,Victim,[Victim]
1379,रूस,"रूस-यूक्रेन युद्ध, इजरायल-हमास की जंग और ईरान-...",Protagonist,Peacemaker,[Peacemaker]
...,...,...,...,...,...
581,Владимир Путин,АР: Путин изпрати мощен сигнал на Запада с пос...,Antagonist,Instigator,[Instigator]
1050,रूस,रूस में रविवार (23 जून 2024) को दक्षिणी प्रांत...,Innocent,Victim,[Victim]
1552,गाजा,रूस-यूक्रेन युद्ध के दो साल: क्या भारत की विदे...,Innocent,Victim,[Victim]
76,Russian military,Kyiv's Mayor Says He's 'Ready to Fight' as Rus...,Antagonist,Instigator,[Instigator]


In [ ]:
# Classify the entity_mentioned in the article into the main role labels: Antagonist, Protagonist, Innocent.
#             if the main role label is Antagonist, classify the entitity_mentioned into fine grain role labels: Instigator, Conspirator, Tyrant, Foreign Adversary, Traitor, Spy, Saboteur, Corrupt, Incompetent, Terrorist, Deceiver, Bigot.
#             if the main role label is Protagonist, classify the entitity_mentioned into fine grain role labels: Guardian,Martyr,Peacemaker,Rebel,Underdog, and Virtuous.
#             if the main role label is Innocent, classify the entitity_mentioned into fine grain role labels: Forgotten, Exploited, Victim, Scapegoat.

#             There is only one main role label per entitity_mentioned. There may be more than one fine grain role label per entitity_mentioned.

#             The entity_mentioned final label is main role label followed by fine_grain_role_labels.

#             Return the answer as the corresponding final labels



# Classify the entity_mentioned in the article into the main_role_labels: Antagonist, Protagonist, Innocent, and return the answer as the main_role_label,
#             if the main_role_label is Antagonist, classify the entitity_mentioned into fine grain role labels: Instigator, Conspirator, Tyrant, Foreign Adversary, Traitor, Spy, Saboteur, Corrupt, Incompetent, Terrorist, Deceiver, Bigot, and return the answer as the fine_grain_role_label.
#             if the main_role_label is Protagonist, classify the entitity_mentioned into fine grain role labels: Guardian,Martyr,Peacemaker,Rebel,Underdog, and Virtuous, and return the answer as the fine_grain_role_label.
#             if main_role_label is Innocent, classify the entitity_mentioned into fine grain role labels: Forgotten, Exploited, Victim, Scapegoat, and return the answer as the fine_grain_role_label.

In [ ]:
# Define the prompt generation functions
def generate_prompt(data_point):
    return f"""

            Classify the entity_mentioned in the article into  Antagonist, Protagonist, Innocent,
            Instigator, Conspirator, Tyrant, Foreign Adversary, Traitor, Spy, Saboteur, Corrupt, Incompetent, Terrorist, Deceiver, Bigot,
            Guardian, Martyr, Peacemaker, Rebel, Underdog, Virtuous,
            Forgotten, Exploited, Victim, Scapegoat
            and return the answer as the corresponding labels

entity_mentioned:{data_point["entity_mention"]}
text: {data_point["article"]}
label: {data_point["main_role"]}, {data_point["fine-grained_roles"]}


""".strip()

def generate_test_prompt(data_point):
    return f"""

            Classify the entity_mentioned in the article into  Antagonist, Protagonist, Innocent,
            Instigator, Conspirator, Tyrant, Foreign Adversary, Traitor, Spy, Saboteur, Corrupt, Incompetent, Terrorist, Deceiver, Bigot,
            Guardian, Martyr, Peacemaker, Rebel, Underdog, Virtuous,
            Forgotten, Exploited, Victim, Scapegoat
            and return the answer as the corresponding labels

entity_mentioned:    {data_point["entity_mention"]}
text: {data_point["article"]}
label:
 """.strip()

# Generate prompts for training
X_train.loc[:,'text'] = X_train.apply(generate_prompt, axis=1)
X_val_.loc[:,'text'] = X_val_.apply(generate_prompt, axis=1)

# Generate test prompts and extract true labels
y_true = X_val_[['main_role','fine-grained_roles_list']].apply(lambda x: [x['main_role']]+x['fine-grained_roles_list'], axis=1)
X_val = pd.DataFrame(X_val_.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:

# Convert to datasets
train_data = Dataset.from_pandas(X_train[["text"]])
val_data = Dataset.from_pandas(X_val[["text"]])

In [ ]:
train_data['text'][2]

'Classify the entity_mentioned in the article into  Antagonist, Protagonist, Innocent,\n            Instigator, Conspirator, Tyrant, Foreign Adversary, Traitor, Spy, Saboteur, Corrupt, Incompetent, Terrorist, Deceiver, Bigot,\n            Guardian, Martyr, Peacemaker, Rebel, Underdog, Virtuous,\n            Forgotten, Exploited, Victim, Scapegoat\n            and return the answer as the corresponding labels\n\nentity_mentioned:Hamas\ntext: The World Needs Peacemaker Trump Again \n\n by Jeff Crouere, The Liberty Daily:\n\nThe world is in total chaos after 39 months of the Biden presidency. The southern border of our country is porous and millions of individuals from around the world have descended on our country.\n\nThese “undocumented migrants” include terrorists, drug dealers, and intelligence agents of countries such as our enemy, China. It should alarm every American that 22,233 Chinese nationals have illegally entered the United States since the beginning of the fiscal year in Oct

In [ ]:
val_data['text'][2]

'Classify the entity_mentioned in the article into  Antagonist, Protagonist, Innocent,\n            Instigator, Conspirator, Tyrant, Foreign Adversary, Traitor, Spy, Saboteur, Corrupt, Incompetent, Terrorist, Deceiver, Bigot,\n            Guardian, Martyr, Peacemaker, Rebel, Underdog, Virtuous,\n            Forgotten, Exploited, Victim, Scapegoat\n            and return the answer as the corresponding labels\n\nentity_mentioned:    Comissão Europeia\ntext: UE transfere mais de 9 mil milhões de dólares em ativos russos congelados para a Ucrânia\n\nA presidente da Comissão Europeia, Ursula von der Leyen, afirmou que a União Europeia vai transferir 1,5 mil milhões de euros em ativos russos congelados para a Ucrânia esta sexta-feira, 26.\n\nA decisão surge depois de os líderes do G7 e da UE terem acordado no mês passado utilizar 50 mil milhões de dólares dos lucros dos ativos russos congelados nos bancos europeus para financiar a compra de armas e a reconstrução do país em guerra.\n\n“Hoje

In [ ]:
base_model_name ="meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

model = LlamaForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype="float16",
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

tokenizer.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
def predict(test, model, tokenizer):

    y_pred_fine_role=[]
    y_pred = []
    categories = ["Antagonist", "Protagonist", "Innocent"]
    taxonomy = {"Protagonist":['Guardian','Martyr','Peacemaker','Rebel','Underdog', 'Virtuous'],
                "Antagonist":['Instigator', 'Conspirator', 'Tyrant', 'Foreign Adversary', 'Traitor', 'Spy', 'Saboteur', 'Corrupt', 'Incompetent', 'Terrorist', 'Deceiver', 'Bigot'],
                "Innocent":['Forgotten', 'Exploited', 'Victim', 'Scapegoat']}

    fg_labels = ['Guardian','Martyr','Peacemaker','Rebel','Underdog', 'Virtuous',
                'Instigator', 'Conspirator', 'Tyrant', 'Foreign Adversary', 'Traitor', 'Spy', 'Saboteur', 'Corrupt', 'Incompetent', 'Terrorist', 'Deceiver', 'Bigot',
                'Forgotten', 'Exploited', 'Victim', 'Scapegoat']

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        pipe = pipeline(task="text-generation",
                        model=model,
                        tokenizer=tokenizer,
                        max_new_tokens=20,
                        temperature=0.1)



        result = pipe(prompt)

        answer = result[0]['generated_text'].split("label:")[-1].strip()

        # Determine the predicted category
        for category in categories:
            if category.lower() in answer.lower():
              y_pred_main_role=category

            else:
              y_pred_main_role="none"

            break

        y_pred_fine_role = []
        for cat in fg_labels :

            if cat.lower() in answer.lower():

              y_pred_fine_role.append(cat)

        if y_pred_main_role!="none":
          y_pred_=[y_pred_main_role]+y_pred_fine_role
        else:
          y_pred_ = y_pred_fine_role


        y_pred.append(y_pred_)

    return y_pred

y_pred = predict(X_val, model, tokenizer)

100%|██████████| 507/507 [10:38<00:00,  1.26s/it]


In [ ]:
def evaluate(y_true, y_pred):
    mlb = MultiLabelBinarizer()
    y_true_bi=mlb.fit_transform(y_true)
    y_pred_bi=mlb.transform(y_pred)


    # Generate classification report
    class_report = classification_report(y_true_bi,  y_pred_bi, target_names=list(mlb.classes_))
    print('\nClassification Report:')
    print(class_report)

    # # Generate confusion matrix
    # conf_matrix = confusion_matrix(y_true_bi,  y_pred_bi, labels=list(mlb.classes_))
    # print('\nConfusion Matrix:')
    # print(conf_matrix)

evaluate(y_true, y_pred)


Classification Report:
                   precision    recall  f1-score   support

       Antagonist       0.42      0.13      0.20       233
            Bigot       0.00      0.00      0.00         5
      Conspirator       0.00      0.00      0.00        25
          Corrupt       0.00      0.00      0.00        12
         Deceiver       0.25      0.07      0.11        15
        Exploited       0.00      0.00      0.00         9
Foreign Adversary       0.23      0.26      0.25        69
        Forgotten       0.00      0.00      0.00         3
         Guardian       0.33      0.03      0.06        64
      Incompetent       0.12      0.04      0.06        28
         Innocent       0.00      0.00      0.00       117
       Instigator       0.11      0.10      0.11        48
           Martyr       0.00      0.00      0.00         3
       Peacemaker       0.33      0.06      0.11        31
      Protagonist       0.00      0.00      0.00       157
            Rebel       0.00   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)
modules = find_all_linear_names(model)
modules

['v_proj', 'down_proj', 'o_proj', 'up_proj', 'gate_proj', 'k_proj', 'q_proj']

In [ ]:
output_dir="/content/drive/MyDrive/Datasci_266_NLP/Final_Project/llama-3.2-3B-fine-tuned-model_all_roles"

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM", #'LlamaForCausalLM'
    target_modules=modules,
)

training_arguments = SFTConfig(
    output_dir=output_dir,                    # directory to save and repository id
    num_train_epochs=15,                       # number of training epochs
    per_device_train_batch_size=16,            # batch size per device during training
    gradient_accumulation_steps=8,            # number of steps before performing a backward/update pass
    gradient_checkpointing=True,              # use gradient checkpointing to save memory
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,                       # learning rate, based on QLoRA paper
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,                        # max gradient norm based on QLoRA paper
    max_steps=-1,
    warmup_ratio=0.03,                        # warmup ratio based on QLoRA paper
    group_by_length=False,
    lr_scheduler_type="cosine",               # use cosine learning rate scheduler
    report_to="tensorboard",                  # report metrics to "tensorboard"
    eval_strategy="steps",              # save checkpoint every epoch
    eval_steps = 0.2,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
    dataset_kwargs={
    "add_special_tokens": False,
    "append_concat_token": False}
)



trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    tokenizer=tokenizer

)

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
45,1.323600,1.358219
90,0.907300,0.934168
135,0.527500,0.688959
180,0.445000,0.596437
225,0.294900,0.580919


TrainOutput(global_step=225, training_loss=0.8465732043319278, metrics={'train_runtime': 3667.188, 'train_samples_per_second': 8.295, 'train_steps_per_second': 0.061, 'total_flos': 2.579892667416576e+17, 'train_loss': 0.8465732043319278, 'epoch': 14.503937007874015})

In [ ]:
output_dir="/content/drive/MyDrive/Datasci_266_NLP/Final_Project/llama-3.2-3B-fine-tuned-model_all_roles"

In [ ]:
log_dir = output_dir+"/runs"

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {log_dir}

In [ ]:
output_dir

'/content/drive/MyDrive/Datasci_266_NLP/Final_Project/llama-3.2-3B-fine-tuned-model_all_roles'

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

('/content/drive/MyDrive/Datasci_266_NLP/Final_Project/llama-3.2-3B-fine-tuned-model_all_roles/tokenizer_config.json',
 '/content/drive/MyDrive/Datasci_266_NLP/Final_Project/llama-3.2-3B-fine-tuned-model_all_roles/special_tokens_map.json',
 '/content/drive/MyDrive/Datasci_266_NLP/Final_Project/llama-3.2-3B-fine-tuned-model_all_roles/tokenizer.json')

In [ ]:
base_model_name =  output_dir #path/to/your/model/or/name/on/hub"
adapter_model_name =  output_dir

model = AutoModelForCausalLM.from_pretrained(base_model_name)
model = PeftModel.from_pretrained(model, adapter_model_name)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def predict(test, model, tokenizer):
    y_pred_fine_role=[]
    y_pred = []
    categories = ["Antagonist", "Protagonist", "Innocent"]
    taxonomy = {"Protagonist":['Guardian','Martyr','Peacemaker','Rebel','Underdog', 'Virtuous'],
                "Antagonist":['Instigator', 'Conspirator', 'Tyrant', 'Foreign Adversary', 'Traitor', 'Spy', 'Saboteur', 'Corrupt', 'Incompetent', 'Terrorist', 'Deceiver', 'Bigot'],
                "Innocent":['Forgotten', 'Exploited', 'Victim', 'Scapegoat']}

    fg_labels = ['Guardian','Martyr','Peacemaker','Rebel','Underdog', 'Virtuous',
                'Instigator', 'Conspirator', 'Tyrant', 'Foreign Adversary', 'Traitor', 'Spy', 'Saboteur', 'Corrupt', 'Incompetent', 'Terrorist', 'Deceiver', 'Bigot',
                'Forgotten', 'Exploited', 'Victim', 'Scapegoat']

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]

        pipe = pipeline(task='text-generation',
                        model=model,
                        tokenizer=tokenizer,
                        max_new_tokens=20,
                        temperature=0.1)
                        #,device='cuda')


        result = pipe(prompt)

        answer = result[0]['generated_text'].split("label:")[-1].strip()

        # Determine the predicted category
        for category in categories:
            if category.lower() in answer.lower():
              y_pred_main_role=category

            else:
              y_pred_main_role="none"

            break

        y_pred_fine_role = []
        for cat in fg_labels :

            if cat.lower() in answer.lower():

              y_pred_fine_role.append(cat)

        if y_pred_main_role!="none":
          y_pred_=[y_pred_main_role]+y_pred_fine_role
        else:
          y_pred_ = y_pred_fine_role


        y_pred.append(y_pred_)

    return y_pred

y_pred = predict(X_val, model, tokenizer)

100%|██████████| 507/507 [19:03<00:00,  2.26s/it]


In [ ]:
def evaluate(y_true, y_pred):
    mlb = MultiLabelBinarizer()
    y_true_bi=mlb.fit_transform(y_true)
    y_pred_bi=mlb.transform(y_pred)


    # Generate classification report
    class_report = classification_report(y_true_bi,  y_pred_bi, target_names=list(mlb.classes_))
    print('\nClassification Report:')
    print(class_report)

    # # Generate confusion matrix
    # conf_matrix = confusion_matrix(y_true_bi,  y_pred_bi, labels=list(mlb.classes_))
    # print('\nConfusion Matrix:')
    # print(conf_matrix)

evaluate(y_true, y_pred)


Classification Report:
                   precision    recall  f1-score   support

       Antagonist       0.53      0.53      0.53       233
            Bigot       0.00      0.00      0.00         5
      Conspirator       0.10      0.24      0.14        25
          Corrupt       0.04      0.25      0.07        12
         Deceiver       0.02      0.07      0.03        15
        Exploited       0.00      0.00      0.00         9
Foreign Adversary       0.14      0.07      0.10        69
        Forgotten       0.00      0.00      0.00         3
         Guardian       0.14      0.03      0.05        64
      Incompetent       0.08      0.54      0.14        28
         Innocent       0.00      0.00      0.00       117
       Instigator       0.07      0.10      0.09        48
           Martyr       0.00      0.00      0.00         3
       Peacemaker       0.09      0.13      0.11        31
      Protagonist       0.00      0.00      0.00       157
            Rebel       0.02   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
